## §0 Setup and Methodology

**Session 3b — DL Experiment Pass.** This notebook trains MLP, LSTM, and Transformer sequence models on the 29-asset 2003–2026
universe, wraps predictions into `MSR(μ̂)` and `SignalTilt(μ̂)` strategies, builds the headline `MSR(EnsembleDL_μ̂)` ensemble,
and merges 7 new rows into the Session 2 master comparison (28 rows → 35 rows). Sharpe bar to beat: **2.579** (MSR(Ensemble_μ̂)).

In [ ]:
import os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib
matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10})
warnings.filterwarnings('ignore')
os.environ.setdefault('OMP_NUM_THREADS', '1')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))

from _shared import ann_sharpe, ann_return, ann_vol, max_drawdown, apply_vmp_overlay, TRADING_DAYS
from aiam.data.panel import Panel
from aiam.data.split import TRAIN_END, TEST_START
from aiam.features.technical import (
    momentum, volatility, rsi, atr, bollinger, gap, volume_signal, forward_returns
)
from aiam.features.asset_class import asset_class_one_hot
from aiam.ml.workflow import (
    chronological_splits, leakage_check_forward_returns, apply_standardizer, cross_sectional_score
)
from aiam.estimators.covariance import ledoit_wolf_cov
from aiam.dl.workflow import (
    fit_mlp_regressor, fit_lstm_regressor, fit_transformer_regressor,
    fit_with_seed_ensemble, build_sequence_windows, set_global_seed, _predict
)
from aiam.strategy.dl_strategies import (
    MLPSignalStrategy, LSTMSignalStrategy, TransformerSignalStrategy, EnsembleDLSignalStrategy
)

import torch
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

print(f'ROOT: {ROOT}')
print(f'Train end: {TRAIN_END}  |  Test start: {TEST_START}')
print(f'Device: {DEVICE}  (torch {torch.__version__})')

**Methodology note.** DL models are trained once on the full training window (2003–2019, ~17 years) and evaluated
out-of-sample on 2023–2026. Validation window is the last 15% of the pre-test period (~2019-10 → 2022-12) and is
used for early stopping only. Single-fit methodology: no walk-forward refit (Session 2 Finding #5). Multi-seed
ensemble (5 seeds) provides stability. Sharpe bar: **2.579** from `MSR(Ensemble_μ̂)` (Session 2).

In [ ]:
# Load returns and prices
returns = pd.read_parquet(ROOT / 'data/cache/returns_29_2003_2026.parquet')
returns.index = pd.to_datetime(returns.index)
returns.index.name = 'Date'
returns.columns.name = 'Asset'

prices = pd.read_parquet(ROOT / 'data/cache/prices_29.parquet')
prices.index = pd.to_datetime(prices.index)
prices.index.name = 'Date'
prices.columns.name = 'Asset'

# Published strategy returns for paper baselines
base_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_base.parquet')
base_pub.index = pd.to_datetime(base_pub.index)
vmp_pub = pd.read_parquet(ROOT / 'data/published/strategy_returns_vmp.parquet')
vmp_pub.index = pd.to_datetime(vmp_pub.index)

# SWITCH(v2a) — cached from Session 2
sw_oos = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet')
sw_oos.index = pd.to_datetime(sw_oos.index)
switch_v2a = sw_oos['SWITCH_v2a_train_only'].rename('SWITCH(v2a)')

# Session 2 ML return series (test period) — for comparison plots
ml_returns_ext = pd.read_parquet(ROOT / 'data/cache/portfolio_returns/ml_strategies_29assets_extended.parquet')

print(f'Returns: {returns.shape}, {returns.index[0].date()} to {returns.index[-1].date()}')
print(f'Prices: {prices.shape}')
print(f'Published base strategies: {base_pub.shape[1]}')
print(f'ML extended returns (test): {ml_returns_ext.shape}')

## §1 Feature Engineering for DL

Same 17-feature panel as Session 2 (10 numeric + 7 asset-class one-hots). This ensures the DL comparison is
apples-to-apples with the ML strategies — identical features, identical target, identical data.

In [ ]:
# Load OHLCV, unstack to wide format
ohlcv_raw = pd.read_parquet(ROOT / 'data/cache/prices_29_ohlcv_2003_2026.parquet')
ohlcv_raw.index = pd.MultiIndex.from_tuples(
    [(pd.Timestamp(d), t) for d, t in ohlcv_raw.index],
    names=['Date', 'Asset']
)

prices_close  = ohlcv_raw['close'].unstack('Asset')
prices_open   = ohlcv_raw['open'].unstack('Asset')
prices_high   = ohlcv_raw['high'].unstack('Asset')
prices_low    = ohlcv_raw['low'].unstack('Asset')
prices_volume = ohlcv_raw['volume'].unstack('Asset')

rets = returns.copy()
ohlc_dict = {'open': prices_open, 'high': prices_high, 'low': prices_low,
             'close': prices_close, 'volume': prices_volume}

print(f'prices_close: {prices_close.shape}, volume: {prices_volume.shape}')

In [ ]:
# 10 numeric features (identical to Session 2)
feat_mom_21   = momentum(rets, 21)
feat_mom_63   = momentum(rets, 63)
feat_mom_252  = momentum(rets, 252)
feat_vol_60   = volatility(rets, 60)
feat_vol_252  = volatility(rets, 252)
feat_rsi_14   = rsi(prices_close, 14)
feat_atr_raw  = atr(ohlc_dict, 14)
feat_atr_ratio = feat_atr_raw / prices_close
bb = bollinger(prices_close, window=20)
feat_bb_pct   = bb['pct']
feat_gap      = gap(ohlc_dict)
feat_vol_sig  = volume_signal(prices_volume, lookback=21)

numeric_frames = {
    'mom_21':        feat_mom_21,
    'mom_63':        feat_mom_63,
    'mom_252':       feat_mom_252,
    'vol_60':        feat_vol_60,
    'vol_252':       feat_vol_252,
    'rsi_14':        feat_rsi_14,
    'atr_14_ratio':  feat_atr_ratio,
    'bb_pct':        feat_bb_pct,
    'gap':           feat_gap,
    'vol_signal_21': feat_vol_sig,
}
panel_numeric = pd.concat({k: v.stack() for k, v in numeric_frames.items()}, axis=1)
panel_numeric.index.names = ['Date', 'Asset']

# 7 asset-class one-hots
assets = rets.columns.tolist()
oh = asset_class_one_hot(assets)
feature_panel = panel_numeric.join(oh, on='Asset')
feature_panel = feature_panel.dropna(subset=['mom_252', 'vol_252'])

NUMERIC_COLS = list(numeric_frames.keys())
AC_COLS = list(oh.columns)
FEATURE_COLS = NUMERIC_COLS + AC_COLS

print(f'Feature panel: {feature_panel.shape}')
print(f'Feature cols ({len(FEATURE_COLS)}): {FEATURE_COLS}')

## §2 Target Construction and Leakage Verification

Target: 21-day forward simple return (same horizon as Session 2 ML). Leakage-verified via spot-check.

In [ ]:
HORIZON = 21
fwd_ret_wide = forward_returns(rets, HORIZON)
target_panel = fwd_ret_wide.stack()
target_panel.index.names = ['Date', 'Asset']
target_panel.name = 'target_21d'

common_idx = feature_panel.index.intersection(target_panel.dropna().index)
X_full = feature_panel.loc[common_idx]
y_full = target_panel.loc[common_idx]

print(f'Target panel: {target_panel.shape}, X_full: {X_full.shape}, y_full: {y_full.shape}')

In [ ]:
# Spot-check 3 (asset, date) pairs for look-ahead leakage
check_pairs = [
    ('AAPL.US', pd.Timestamp('2010-06-15')),
    ('SPY.US',  pd.Timestamp('2015-03-20')),
    ('GLD.US',  pd.Timestamp('2019-11-01')),
]
for asset, date in check_pairs:
    ok = leakage_check_forward_returns(rets, fwd_ret_wide, HORIZON, asset, date)
    print(f'{asset} @ {date.date()}: leakage_check = {"PASS" if ok else "FAIL"}')

## §3 Train/Validation/Test Splits

Paper-locked boundaries from `aiam.data.split`: train through 2022-12-31, test from 2023-01-01.
Validation is the last 15% of the pre-test period (~2019-10 → 2022-12), used for early stopping only.

In [ ]:
panel_dates = X_full.index.get_level_values('Date').unique().sort_values()
train_dates, val_dates, test_dates = chronological_splits(
    panel_dates, train_end=TRAIN_END, test_start=TEST_START, validation_share=0.15
)

print(f'Train:      {train_dates[0].date()} → {train_dates[-1].date()}  ({len(train_dates)} days, {len(train_dates)*29:,} obs)')
print(f'Validation: {val_dates[0].date()} → {val_dates[-1].date()}  ({len(val_dates)} days, {len(val_dates)*29:,} obs)')
print(f'Test:       {test_dates[0].date()} → {test_dates[-1].date()}  ({len(test_dates)} days, {len(test_dates)*29:,} obs)')

In [ ]:
# Strategy family/color config — extended for DL
STRATEGY_FAMILY = {
    'EW': 'Classical', 'MSR(LW)': 'Classical', 'SWITCH(v2a)': 'Classical', 'VMP(MDP(LW))': 'Classical',
    'SignalTilt(mom_252)': 'Classical',
    'SignalTilt(Lasso)': 'ML (single-fit)', 'SignalTilt(RF)': 'ML (single-fit)', 'SignalTilt(XGB)': 'ML (single-fit)',
    'MSR(Lasso_\u03bc\u0302)': 'ML (single-fit)', 'MSR(RF_\u03bc\u0302)': 'ML (single-fit)', 'MSR(XGB_\u03bc\u0302)': 'ML (single-fit)',
    'VMP(SignalTilt(Lasso))': 'ML + VMP', 'VMP(SignalTilt(RF))': 'ML + VMP', 'VMP(SignalTilt(XGB))': 'ML + VMP',
    'VMP(MSR(Lasso_\u03bc\u0302))': 'ML + VMP', 'VMP(MSR(RF_\u03bc\u0302))': 'ML + VMP', 'VMP(MSR(XGB_\u03bc\u0302))': 'ML + VMP',
    'SignalTilt(Ensemble)': 'ML (ensemble)', 'MSR(Ensemble_\u03bc\u0302)': 'ML (ensemble)',
    'MSR(MLP_\u03bc\u0302)': 'DL (single-fit)', 'MSR(LSTM_\u03bc\u0302)': 'DL (single-fit)', 'MSR(Transformer_\u03bc\u0302)': 'DL (single-fit)',
    'SignalTilt(MLP)': 'DL (single-fit)', 'SignalTilt(LSTM)': 'DL (single-fit)', 'SignalTilt(Transformer)': 'DL (single-fit)',
    'MSR(EnsembleDL_\u03bc\u0302)': 'DL (ensemble)',
}
FAMILY_COLORS = {
    'Classical':                   '#1f4e79',
    'ML (single-fit)':             '#d62728',
    'ML + VMP':                    '#ff7f0e',
    'ML (ensemble)':               '#2ca02c',
    'DL (single-fit)':             '#9467bd',
    'DL (ensemble)':               '#e377c2',
    'Walk-forward (default HPs)':  '#7f7f7f',
    'Walk-forward (val-optimal HPs)': '#bcbd22',
}
print('Families:', list(FAMILY_COLORS.keys()))

## §4 MLP Training — 5 Seeds

Architecture: 2-layer MLP, hidden_dims=(32, 16), dropout=0.10. Input: 17 cross-sectional features (tabular, no sequence).
1,121 parameters. max_epochs=120, patience=15. Device: auto-detected (MPS on M4 Mac).

In [ ]:
print('=== MLP Training (5 seeds) ===')
t_mlp_start = time.time()

mlp_strat = MLPSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    seeds=(0, 1, 2, 3, 4),
    hidden_dims=(32, 16),
    dropout=0.10,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    max_epochs=120,
    patience=15,
    device=DEVICE,
)

t_mlp = time.time() - t_mlp_start
print(f'MLP training done in {int(t_mlp//60)}:{int(t_mlp%60):02d}')
print('\nPer-seed summaries:')
for fr in mlp_strat._seed_ensemble.fits:
    s = fr.summary
    print(f'  seed={s["seed"]}  best_epoch={s["best_epoch"]:3d}  val_MSE={s["best_val_loss"]:.6f}  val_rank_IC={s["val_ic_at_best"]:+.4f}  n_epochs={s["n_epochs_trained"]}')

## §5 LSTM Training — 5 Seeds

Architecture: single-layer LSTM, hidden_dim=24, dropout=0.10. Input shape: (batch, 63, 17) — 63-day lookback sequence.
4,153 parameters. max_epochs=80, patience=12. Sequences are per-asset (no cross-asset mixing at this stage).

In [ ]:
print('=== LSTM Training (5 seeds) ===')
t_lstm_start = time.time()

lstm_strat = LSTMSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    seeds=(0, 1, 2, 3, 4),
    hidden_dim=24,
    dropout=0.10,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    max_epochs=80,
    patience=12,
    device=DEVICE,
)

t_lstm = time.time() - t_lstm_start
print(f'LSTM training done in {int(t_lstm//60)}:{int(t_lstm%60):02d}')
print('\nPer-seed summaries:')
for fr in lstm_strat._seed_ensemble.fits:
    s = fr.summary
    print(f'  seed={s["seed"]}  best_epoch={s["best_epoch"]:3d}  val_MSE={s["best_val_loss"]:.6f}  val_rank_IC={s["val_ic_at_best"]:+.4f}  n_epochs={s["n_epochs_trained"]}')

## §6 Transformer Training — 5 Seeds

Architecture: 2-layer Transformer encoder, d_model=32, nhead=4, dropout=0.10. Input shape: (batch, 63, 17).
42,401 parameters (includes learnable positional embeddings shape (1, 512, 32), std=0.02 init).
**max_epochs=15, patience=6** (CPU/MPS constraint: one epoch ≈ 14s on MPS; early stopping typically exits at epoch 8–12).
Manual MHA+FFN blocks (avoids Apple Silicon segfault with nn.TransformerEncoder).

In [ ]:
print('=== Transformer Training (5 seeds) ===')
t_tfm_start = time.time()

tfm_strat = TransformerSignalStrategy(
    feature_panel=X_full,
    target_panel=y_full,
    feature_cols=FEATURE_COLS,
    train_end=TRAIN_END,
    validation_share=0.15,
    seeds=(0, 1, 2, 3, 4),
    d_model=32,
    nhead=4,
    num_layers=2,
    dropout=0.10,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    max_epochs=15,
    patience=6,
    device=DEVICE,
)

t_tfm = time.time() - t_tfm_start
print(f'Transformer training done in {int(t_tfm//60)}:{int(t_tfm%60):02d}')
print('\nPer-seed summaries:')
for fr in tfm_strat._seed_ensemble.fits:
    s = fr.summary
    print(f'  seed={s["seed"]}  best_epoch={s["best_epoch"]:3d}  val_MSE={s["best_val_loss"]:.6f}  val_rank_IC={s["val_ic_at_best"]:+.4f}  n_epochs={s["n_epochs_trained"]}')

t_total_training = t_mlp + t_lstm + t_tfm
print(f'\nTotal DL training: {int(t_total_training//60)}:{int(t_total_training%60):02d}')
print(f'  MLP:         {int(t_mlp//60)}:{int(t_mlp%60):02d}')
print(f'  LSTM:        {int(t_lstm//60)}:{int(t_lstm%60):02d}')
print(f'  Transformer: {int(t_tfm//60)}:{int(t_tfm%60):02d}')

## §7 Walk-Forward Evaluation Setup

Build Panel, define test dates, and implement the two evaluation wrappers: Approach B (SignalTilt) and Approach A (MSR(μ̂)).
Both are identical to Session 2 notebook 03 — same function signatures, same `lookback_cov=504` for covariance estimation.

In [ ]:
# Build Panel for evaluation
panel = Panel({'prices': prices, 'returns': returns})
eval_dates = returns.index[returns.index >= TEST_START]

def walk_forward_returns(strat, dates, returns_wide, panel_obj):
    """Walk-forward: weights on date t, applied to t+1 return."""
    ret_series = {}
    for d in dates[:-1]:
        w = strat.predict_weights(panel_obj, d)
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

def msr_ml_returns(strat, eval_dates, returns_wide, lookback_cov=504):
    """MSR with DL-predicted mu_hat. Long-only, renormalized."""
    ret_series = {}
    for d in eval_dates[:-1]:
        mu_hat = cross_sectional_score(strat.predictions, d)
        if mu_hat.empty:
            continue
        r_window = returns_wide.loc[returns_wide.index <= d].tail(lookback_cov)
        r_clean = r_window[mu_hat.index].dropna(how='any')
        if len(r_clean) < 60:
            continue
        sigma = ledoit_wolf_cov(r_clean)
        try:
            sigma_inv = np.linalg.inv(sigma)
        except np.linalg.LinAlgError:
            continue
        mu = mu_hat.values
        w_unc = sigma_inv @ mu
        w_lo = np.clip(w_unc, 0, None)
        total = w_lo.sum()
        if total < 1e-12:
            w_final = pd.Series(1.0 / len(mu_hat), index=mu_hat.index)
        else:
            w_final = pd.Series(w_lo / total, index=mu_hat.index)
        next_d_mask = returns_wide.index > d
        if not next_d_mask.any():
            continue
        next_d = returns_wide.index[next_d_mask][0]
        r_next = returns_wide.loc[next_d]
        ret_series[next_d] = (w_final.reindex(r_next.index).fillna(0) * r_next).sum()
    return pd.Series(ret_series)

def metrics(r, name):
    r = r.reindex(test_idx).dropna() if 'test_idx' in dir() else r.dropna()
    return {'Strategy': name, 'Ann Ret': ann_return(r), 'Ann Vol': ann_vol(r),
            'Sharpe': ann_sharpe(r), 'Max DD': max_drawdown(r)}

print(f'eval_dates: {eval_dates[0].date()} to {eval_dates[-1].date()} ({len(eval_dates)} days)')

## §8 Approach B — SignalTilt(DL)

Each DL strategy's `predict_weights` applies a tilt_strength=0.5 cross-sectional score on its cached test predictions.
Walk-forward evaluation identical to Session 2.

In [ ]:
print('Running Approach B — SignalTilt(DL)...')
t0 = time.time()
ret_st_mlp = walk_forward_returns(mlp_strat, eval_dates, returns, panel)
print(f'  SignalTilt(MLP) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_st_lstm = walk_forward_returns(lstm_strat, eval_dates, returns, panel)
print(f'  SignalTilt(LSTM) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_st_tfm = walk_forward_returns(tfm_strat, eval_dates, returns, panel)
print(f'  SignalTilt(Transformer) done ({time.time()-t0:.0f}s)')

# Align test dates
test_mask = ret_st_mlp.index >= TEST_START
test_idx = ret_st_mlp.index[test_mask]

approach_b = pd.DataFrame([
    metrics(ret_st_mlp,  'SignalTilt(MLP)'),
    metrics(ret_st_lstm, 'SignalTilt(LSTM)'),
    metrics(ret_st_tfm,  'SignalTilt(Transformer)'),
]).set_index('Strategy')
print('\nApproach B — Test period metrics:')
print(approach_b.round(3).to_string())

## §9 Approach A — MSR(DL μ̂)

Ledoit-Wolf covariance (504-day trailing window) + unconstrained max-Sharpe weights, clipped to long-only.
Uses each strategy's cached cross-sectional predictions as μ̂.

In [ ]:
print('Running Approach A — MSR(DL \u03bc\u0302)...')
t0 = time.time()
ret_msr_mlp = msr_ml_returns(mlp_strat, eval_dates, returns)
print(f'  MSR(MLP) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_lstm = msr_ml_returns(lstm_strat, eval_dates, returns)
print(f'  MSR(LSTM) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_tfm = msr_ml_returns(tfm_strat, eval_dates, returns)
print(f'  MSR(Transformer) done ({time.time()-t0:.0f}s)')

approach_a = pd.DataFrame([
    metrics(ret_msr_mlp,  'MSR(MLP_\u03bc\u0302)'),
    metrics(ret_msr_lstm, 'MSR(LSTM_\u03bc\u0302)'),
    metrics(ret_msr_tfm,  'MSR(Transformer_\u03bc\u0302)'),
]).set_index('Strategy')
print('\nApproach A — Test period metrics:')
print(approach_a.round(3).to_string())

## §10 Ensemble DL — MSR(EnsembleDL_μ̂)

Equal-weighted average of MLP, LSTM, and Transformer cached predictions → fed into MSR wrapping.
Mirrors Session 2's `MSR(Ensemble_μ̂)` construction (Lasso + RF + XGB). This is the headline DL strategy.

In [ ]:
# Equal-weighted ensemble of MLP + LSTM + Transformer predictions
ensemble_strat = EnsembleDLSignalStrategy(
    strategies=[mlp_strat, lstm_strat, tfm_strat],
    tilt_strength=0.5,
)
print(f'Ensemble predictions: {len(ensemble_strat.predictions)} (Date, Asset) observations')

print('Running Ensemble...')
t0 = time.time()
ret_st_ens = walk_forward_returns(ensemble_strat, eval_dates, returns, panel)
print(f'  SignalTilt(EnsembleDL) done ({time.time()-t0:.0f}s)')

t0 = time.time()
ret_msr_ens = msr_ml_returns(ensemble_strat, eval_dates, returns)
print(f'  MSR(EnsembleDL_\u03bc\u0302) done ({time.time()-t0:.0f}s)')

all_7 = pd.DataFrame([
    metrics(ret_msr_mlp,  'MSR(MLP_\u03bc\u0302)'),
    metrics(ret_msr_lstm, 'MSR(LSTM_\u03bc\u0302)'),
    metrics(ret_msr_tfm,  'MSR(Transformer_\u03bc\u0302)'),
    metrics(ret_st_mlp,   'SignalTilt(MLP)'),
    metrics(ret_st_lstm,  'SignalTilt(LSTM)'),
    metrics(ret_st_tfm,   'SignalTilt(Transformer)'),
    metrics(ret_msr_ens,  'MSR(EnsembleDL_\u03bc\u0302)'),
]).set_index('Strategy')
print('\n7 new DL strategies — Test period metrics (sorted by Sharpe):')
print(all_7.sort_values('Sharpe', ascending=False).round(3).to_string())

## §11 Per-Seed OOS Backtests — Stability Table

For each architecture: extract single-seed predictions for each of the 5 seeds → run MSR(μ̂) backtest →
compute OOS Sharpe. Produces the stability table with val metrics + OOS Sharpe distribution.
15 MSR backtests total (~seconds each).

In [ ]:
def single_seed_predictions(strat, seed_idx):
    """Extract predictions from a single seed of a trained DL strategy."""
    records = {}
    for date in strat._test_dates:
        try:
            X_date = strat._feature_panel.xs(date, level=0)[strat._feature_cols]
        except KeyError:
            continue
        assets = list(X_date.index)
        model = strat._seed_ensemble.fits[seed_idx].model
        if strat._lookback is None:
            X_std = apply_standardizer(X_date, strat._center, strat._scale, strat._feature_cols)
            preds = _predict(model, X_std.values.astype('float32'))
            records[date] = pd.Series(preds, index=assets, name='pred')
        else:
            windows, valid_assets = strat._build_sequence_batch(assets, date)
            if not windows:
                continue
            X_batch = np.stack(windows, axis=0)
            preds = _predict(model, X_batch)
            records[date] = pd.Series(preds, index=valid_assets, name='pred')
    if not records:
        return pd.Series(dtype=float, name='pred')
    pred_series = pd.concat(records)
    pred_series.index.names = ['Date', 'Asset']
    return pred_series

class _PredWrapper:
    def __init__(self, predictions):
        self.predictions = predictions

print('Computing per-seed OOS Sharpes (15 backtests)...')
stability_rows = []
seed_sharpes_all = {}  # for box plot

for name, strat in [('MLP', mlp_strat), ('LSTM', lstm_strat), ('Transformer', tfm_strat)]:
    seed_sharpes = []
    for k in range(5):
        seed_preds = single_seed_predictions(strat, k)
        wrapper = _PredWrapper(seed_preds)
        r = msr_ml_returns(wrapper, eval_dates, returns)
        sh = ann_sharpe(r.reindex(test_idx).dropna())
        seed_sharpes.append(sh)
    seed_sharpes_all[name] = seed_sharpes

    val_losses = [fr.summary['best_val_loss'] for fr in strat._seed_ensemble.fits]
    val_ics = [fr.summary['val_ic_at_best'] for fr in strat._seed_ensemble.fits]
    stability_rows.append({
        'Model': name,
        'Mean val_MSE': np.mean(val_losses),
        'Stdev val_MSE': np.std(val_losses),
        'Mean val_rank_IC': np.mean(val_ics),
        'Stdev val_rank_IC': np.std(val_ics),
        'Min OOS Sharpe': np.min(seed_sharpes),
        'Mean OOS Sharpe': np.mean(seed_sharpes),
        'Max OOS Sharpe': np.max(seed_sharpes),
    })
    print(f'{name}: {[f"{s:.3f}" for s in seed_sharpes]}  mean={np.mean(seed_sharpes):.3f} \u00b1 {np.std(seed_sharpes):.3f}')

stability_table = pd.DataFrame(stability_rows).set_index('Model')
disp_cols = ['Mean val_MSE', 'Stdev val_MSE', 'Mean val_rank_IC', 'Stdev val_rank_IC',
             'Min OOS Sharpe', 'Mean OOS Sharpe', 'Max OOS Sharpe']
print('\nStability table:')
print(stability_table[disp_cols].round(4).to_string())

## §12 Extended Comparison Table (28 → 35 Strategies)

Append 7 DL strategy rows to the Session 2 master comparison (`ml_strategies_extended_v2_comparison.csv`).
Combined table sorted by Sharpe descending. Saved as `dl_strategies_extended_comparison.csv`.

In [ ]:
out_dir = ROOT / 'data/cache/portfolio_returns'
out_dir.mkdir(parents=True, exist_ok=True)

# Load existing 28-row Session 2 comparison
existing = pd.read_csv(out_dir / 'ml_strategies_extended_v2_comparison.csv').set_index('Strategy')
print(f'Existing (Session 2): {len(existing)} strategies')

# Build 7 new DL rows
new_rows = []
for name, r in [
    ('MSR(MLP_\u03bc\u0302)',          ret_msr_mlp),
    ('MSR(LSTM_\u03bc\u0302)',         ret_msr_lstm),
    ('MSR(Transformer_\u03bc\u0302)',  ret_msr_tfm),
    ('SignalTilt(MLP)',         ret_st_mlp),
    ('SignalTilt(LSTM)',        ret_st_lstm),
    ('SignalTilt(Transformer)', ret_st_tfm),
    ('MSR(EnsembleDL_\u03bc\u0302)',   ret_msr_ens),
]:
    r_test = r.reindex(test_idx).dropna()
    new_rows.append({
        'Strategy': name,
        'Ann Ret': ann_return(r_test),
        'Ann Vol': ann_vol(r_test),
        'Sharpe': ann_sharpe(r_test),
        'Max DD': max_drawdown(r_test),
        'Family': 'DL (ensemble)' if 'EnsembleDL' in name else 'DL (single-fit)',
    })

new_df = pd.DataFrame(new_rows).set_index('Strategy')
combined = pd.concat([existing, new_df]).sort_values('Sharpe', ascending=False)
print(f'Combined: {len(combined)} strategies')

combined.to_csv(out_dir / 'dl_strategies_extended_comparison.csv')
print(f'Saved: {out_dir}/dl_strategies_extended_comparison.csv')

print('\nTop 15 strategies by Sharpe:')
print(combined[['Family', 'Ann Ret', 'Ann Vol', 'Sharpe', 'Max DD']].head(15).round(3).to_string())

## §13 Plots

Four plots mirroring Session 2 style: (1) cumulative equity curves (top 5 + EnsembleDL),
(2) underwater drawdown, (3) per-seed Sharpe distribution (box plot), (4) Sharpe vs model complexity.

In [ ]:
fig_dir = ROOT / 'docs/figures/session3'
fig_dir.mkdir(parents=True, exist_ok=True)

# Build comprehensive returns dict for plots
msr_lw_test    = base_pub['MSR(ledoit_wolf)'][base_pub.index >= TEST_START]
sw_test        = switch_v2a[switch_v2a.index >= TEST_START]
vmp_mdp_test   = vmp_pub['VMP(MDP(ledoit_wolf))'][vmp_pub.index >= TEST_START]

comparison_returns = {
    # Classical baselines
    'EW':               ml_returns_ext['EW'],
    'MSR(LW)':          msr_lw_test,
    'SWITCH(v2a)':      sw_test,
    'VMP(MDP(LW))':    vmp_mdp_test,
    # ML headline strategies
    'MSR(Ensemble_\u03bc\u0302)': ml_returns_ext['MSR(Ensemble_\u03bc\u0302)'],
    # DL strategies
    'MSR(MLP_\u03bc\u0302)':          ret_msr_mlp,
    'MSR(LSTM_\u03bc\u0302)':         ret_msr_lstm,
    'MSR(Transformer_\u03bc\u0302)':  ret_msr_tfm,
    'SignalTilt(MLP)':         ret_st_mlp,
    'SignalTilt(LSTM)':        ret_st_lstm,
    'SignalTilt(Transformer)': ret_st_tfm,
    'MSR(EnsembleDL_\u03bc\u0302)':   ret_msr_ens,
}
print(f'comparison_returns: {len(comparison_returns)} strategies')
print(f'Session 3 figures dir: {fig_dir}')

In [ ]:
# Plot 1: Cumulative wealth — top 5 by Sharpe + EnsembleDL headline
top5 = combined.head(5).index.tolist()
must_include = 'MSR(EnsembleDL_\u03bc\u0302)'
plot_strats = top5 if must_include in top5 else top5[:4] + [must_include]

fig, ax = plt.subplots(figsize=(11, 5))
for strat in plot_strats:
    r = comparison_returns.get(strat, pd.Series(dtype=float)).dropna()
    if len(r) == 0:
        continue
    r = r.reindex(test_idx).dropna()
    wealth = (1 + r).cumprod()
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    color = FAMILY_COLORS.get(fam, '#333333')
    lw = 2.2 if 'EnsembleDL' in strat else 1.5
    ax.plot(wealth.index, wealth.values, label=strat, color=color, lw=lw)
ax.set_yscale('log')
ax.set_ylabel('Cumulative wealth (log, $1\u2192)', fontsize=10)
ax.set_title('Cumulative wealth, top strategies — test period 2023\u20132026', fontsize=11)
ax.legend(loc='upper left', fontsize=8, framealpha=0.85)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'cum_wealth_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session3/cum_wealth_top5.png')

In [ ]:
# Plot 2: Underwater drawdown
fig, ax = plt.subplots(figsize=(11, 4))
for strat in plot_strats:
    r = comparison_returns.get(strat, pd.Series(dtype=float)).dropna()
    if len(r) == 0:
        continue
    r = r.reindex(test_idx).dropna()
    wealth = (1 + r).cumprod()
    dd = wealth / wealth.cummax() - 1
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    color = FAMILY_COLORS.get(fam, '#333333')
    ax.fill_between(dd.index, dd.values, 0, alpha=0.15, color=color)
    ax.plot(dd.index, dd.values, label=strat, color=color, lw=1.3)
ax.set_ylabel('Drawdown', fontsize=10)
ax.set_title('Underwater drawdown, top strategies — test period 2023\u20132026', fontsize=11)
ax.legend(loc='lower left', fontsize=8, framealpha=0.85)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'drawdown_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session3/drawdown_top5.png')

In [ ]:
# Plot 3: Per-seed OOS Sharpe distribution (box + strip)
fig, ax = plt.subplots(figsize=(7, 4))
arch_names = ['MLP', 'LSTM', 'Transformer']
arch_colors = ['#9467bd', '#9467bd', '#9467bd']

positions = range(len(arch_names))
for i, name in enumerate(arch_names):
    sharpes = seed_sharpes_all[name]
    bp = ax.boxplot(sharpes, positions=[i], widths=0.4,
                   patch_artist=True, notch=False,
                   boxprops=dict(facecolor='#c3a6e3', alpha=0.6),
                   medianprops=dict(color='#9467bd', lw=2),
                   whiskerprops=dict(color='#9467bd'),
                   capprops=dict(color='#9467bd'),
                   flierprops=dict(marker='o', color='#9467bd', alpha=0.5))
    # Strip (jitter)
    jitter = np.random.default_rng(42).uniform(-0.08, 0.08, len(sharpes))
    ax.scatter([i + j for j in jitter], sharpes, color='#9467bd', s=30, zorder=3, alpha=0.9)
    # Ensemble mean reference
    ens_sharpe = ann_sharpe(ret_msr_ens.reindex(test_idx).dropna())

# Reference lines
ax.axhline(2.579, color='#2ca02c', lw=1.5, ls='--', label='ML bar (MSR(Ensemble_\u03bc\u0302)) = 2.579')
ax.axhline(ens_sharpe, color='#e377c2', lw=1.5, ls='-.',
           label=f'MSR(EnsembleDL_\u03bc\u0302) = {ens_sharpe:.3f}')

ax.set_xticks(list(positions))
ax.set_xticklabels(arch_names, fontsize=10)
ax.set_ylabel('OOS Sharpe (per seed, MSR wrapping)', fontsize=10)
ax.set_title('Per-seed OOS Sharpe stability — 5 seeds each', fontsize=11)
ax.legend(fontsize=8, framealpha=0.85)
ax.grid(True, alpha=0.3, axis='y')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'per_seed_sharpe.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session3/per_seed_sharpe.png')

In [ ]:
# Plot 4: Sharpe vs model complexity (param count)
model_params = {
    'MSR(MLP_\u03bc\u0302)':          1_121,
    'MSR(LSTM_\u03bc\u0302)':         4_153,
    'MSR(Transformer_\u03bc\u0302)':  42_401,
    'SignalTilt(MLP)':         1_121,
    'SignalTilt(LSTM)':        4_153,
    'SignalTilt(Transformer)': 42_401,
    'MSR(EnsembleDL_\u03bc\u0302)':   1_121 + 4_153 + 42_401,
}
# Add ML reference points (approx, from Session 2)
ml_params = {'MSR(Ensemble_\u03bc\u0302)': 3_000, 'MSR(RF_\u03bc\u0302)': 50_000}
all_params = {**model_params, **ml_params}

fig, ax = plt.subplots(figsize=(9, 5))
for strat, n_params in all_params.items():
    sharpe = combined.loc[strat, 'Sharpe'] if strat in combined.index else None
    if sharpe is None:
        continue
    fam = STRATEGY_FAMILY.get(strat, 'Classical')
    color = FAMILY_COLORS.get(fam, '#333333')
    marker = '*' if 'EnsembleDL' in strat else ('D' if 'Ensemble_\u03bc\u0302' == strat.replace('MSR(', '').replace(')', '') else 'o')
    size = 200 if 'Ensemble' in strat else 60
    ax.scatter(n_params, sharpe, color=color, marker='*' if 'EnsembleDL' in strat else 'o',
               s=size, zorder=3, alpha=0.9)
    ax.annotate(strat.replace('\u03bc\u0302', '\u03bc\u0302'), (n_params, sharpe),
                fontsize=6, xytext=(4, 3), textcoords='offset points')
ax.set_xscale('log')
ax.axhline(2.579, color='#2ca02c', lw=1, ls='--', alpha=0.6, label='ML bar = 2.579')
handles = [mpatches.Patch(color=FAMILY_COLORS[f], label=f)
           for f in ['ML (ensemble)', 'DL (single-fit)', 'DL (ensemble)'] if f in FAMILY_COLORS]
ax.legend(handles=handles, fontsize=8, loc='lower right')
ax.set_xlabel('Model parameter count (log scale)', fontsize=10)
ax.set_ylabel('OOS Sharpe (test 2023\u20132026)', fontsize=10)
ax.set_title('Sharpe vs model complexity — DL vs ML', fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(fig_dir / 'sharpe_vs_complexity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: docs/figures/session3/sharpe_vs_complexity.png')

## §14 Save Artifacts

In [ ]:
# Save DL return series (test period)
dl_returns = pd.DataFrame({
    'MSR(MLP_\u03bc\u0302)':          ret_msr_mlp,
    'MSR(LSTM_\u03bc\u0302)':         ret_msr_lstm,
    'MSR(Transformer_\u03bc\u0302)':  ret_msr_tfm,
    'SignalTilt(MLP)':         ret_st_mlp,
    'SignalTilt(LSTM)':        ret_st_lstm,
    'SignalTilt(Transformer)': ret_st_tfm,
    'MSR(EnsembleDL_\u03bc\u0302)':   ret_msr_ens,
}).reindex(test_idx)
dl_returns.index.name = 'Date'
dl_returns.to_parquet(out_dir / 'dl_strategies_29assets.parquet')
print(f'DL returns: {dl_returns.shape} \u2192 {out_dir}/dl_strategies_29assets.parquet')

# Combined extended comparison already saved in §12
print(f'Extended comparison: {out_dir}/dl_strategies_extended_comparison.csv ({len(combined)} rows)')

# Stability table
stability_table.to_csv(out_dir / 'dl_stability_table.csv')
print(f'Stability table: {out_dir}/dl_stability_table.csv')

print(f'Figures: {fig_dir}/')
for f in sorted(fig_dir.glob('*.png')):
    print(f'  {f.name}')

## §15 Self-Verification Block

In [ ]:
# === Session 3b self-verification ===
print('=' * 60)
print('Session 3b self-verification')
print('=' * 60)

# 1. All seed-ensembles trained
for name, result in [('MLP', mlp_strat._seed_ensemble),
                      ('LSTM', lstm_strat._seed_ensemble),
                      ('Transformer', tfm_strat._seed_ensemble)]:
    assert len(result.fits) == 5, f'{name}: expected 5 seeds, got {len(result.fits)}'
    stab = {'mean': np.mean([f.summary['val_ic_at_best'] for f in result.fits]),
            'stdev': np.std([f.summary['val_ic_at_best'] for f in result.fits])}
    print(f'{name}: 5 seeds, mean val_IC = {stab["mean"]:.4f} \u00b1 {stab["stdev"]:.4f}')

# 2. Stability table is complete
print('\nStability table:')
print(stability_table[disp_cols].round(4).to_string())

# 3. All 7 new strategies present in comparison
new_strats = ['MSR(MLP_\u03bc\u0302)', 'MSR(LSTM_\u03bc\u0302)', 'MSR(Transformer_\u03bc\u0302)',
              'SignalTilt(MLP)', 'SignalTilt(LSTM)', 'SignalTilt(Transformer)',
              'MSR(EnsembleDL_\u03bc\u0302)']
for st in new_strats:
    assert st in combined.index, f'Missing: {st}'
print(f'\nAll 7 DL rows present in comparison table (total rows: {len(combined)})')

# 4. Headline result
headline_sharpe = combined.loc['MSR(EnsembleDL_\u03bc\u0302)', 'Sharpe']
ml_bar = 2.579
verdict = 'CLEARED' if headline_sharpe > ml_bar else 'MISSED'
delta = abs(headline_sharpe - ml_bar)
print(f'\nHeadline: MSR(EnsembleDL_\u03bc\u0302) Sharpe = {headline_sharpe:.3f}  (ML bar: {ml_bar:.3f})')
print(f'DL {verdict} the bar by {delta:.3f}')

# 5. Artifacts on disk
assert (out_dir / 'dl_strategies_29assets.parquet').exists()
assert (out_dir / 'dl_strategies_extended_comparison.csv').exists()
assert (fig_dir / 'cum_wealth_top5.png').exists()
assert (fig_dir / 'per_seed_sharpe.png').exists()
print('\nAll artifacts present on disk: OK')

print('\nSelf-verification: PASS')

## §16 Session 3b Summary

**Wall-clock training** (MPS device, M4 Mac): MLP 2:23 · LSTM 4:06 · Transformer 15:00 · **Total 21:31**

**Stability table** (5 seeds each, MSR wrapping for OOS Sharpe):

| Model | Mean val_MSE | Stdev val_MSE | Mean val_rank_IC | Stdev val_rank_IC | Min OOS Sharpe | Mean OOS Sharpe | Max OOS Sharpe |
|---|---|---|---|---|---|---|---|
| MLP         | 0.0054 | 0.0001 | **0.1345** | 0.0268 | 1.884 | 2.206 | 2.498 |
| LSTM        | 0.0055 | 0.0001 | 0.1048 | 0.0114 | 1.919 | 2.096 | 2.264 |
| Transformer | 0.0054 | 0.0000 | **0.1496** | 0.0297 | 1.799 | 2.114 | 2.397 |

**7 new DL strategy rows** (test period 2023–2026):

| Strategy | Sharpe | Ann Ret | Max DD |
|---|---|---|---|
| MSR(MLP_μ̂)           | **2.356** | 0.224 | -0.091 |
| SignalTilt(MLP)       | 2.306 | 0.660 | -0.276 |
| SignalTilt(Transformer) | 2.277 | 0.506 | -0.218 |
| MSR(EnsembleDL_μ̂)    | 2.175 | 0.241 | -0.129 |
| SignalTilt(LSTM)      | 2.170 | 0.529 | -0.246 |
| MSR(Transformer_μ̂)   | 2.152 | 0.236 | -0.115 |
| MSR(LSTM_μ̂)          | 2.101 | 0.250 | -0.140 |

**Headline:** `MSR(EnsembleDL_μ̂)` Sharpe = **2.175** vs Session 2 bar **2.579** → DL **MISSED** by 0.404.
Best single DL strategy: `MSR(MLP_μ̂)` = 2.356 (ranks 4th in combined 35-strategy comparison).

**Top 10 combined (35-strategy, sorted by Sharpe):**

| Rank | Strategy | Sharpe | Family |
|---|---|---|---|
| 1 | MSR(Ensemble_μ̂)       | 2.579 | ML (ensemble) |
| 2 | VMP(MDP(LW))           | 2.422 | Classical |
| 3 | MSR(RF_μ̂)             | 2.394 | ML (single-fit) |
| 4 | **MSR(MLP_μ̂)**        | **2.356** | **DL (single-fit)** |
| 5 | **SignalTilt(MLP)**    | **2.306** | **DL (single-fit)** |
| 6 | SignalTilt(XGB)        | 2.304 | ML (single-fit) |
| 7 | VMP(SignalTilt(XGB))   | 2.292 | ML + VMP |
| 8 | **SignalTilt(Transformer)** | **2.277** | **DL (single-fit)** |
| 9 | MSR(Lasso_μ̂)          | 2.272 | ML (single-fit) |
| 10| VMP(MSR(Lasso_μ̂))     | 2.255 | ML + VMP |

**Findings:**
1. **MSR wrapping is model-dependent.** MLP: MSR wins (2.356 vs 2.306 SignalTilt). LSTM and Transformer: SignalTilt wins (2.170 vs 2.101; 2.277 vs 2.152). Mirrors Session 2 pattern where well-calibrated predictions (MLP) benefit more from MSR amplification.
2. **Ensemble averaging diluted MLP's signal.** MSR(EnsembleDL_μ̂) = 2.175 < MSR(MLP_μ̂) = 2.356. Averaging with weaker LSTM/Transformer predictions hurt. Contrast with Session 2 where ensemble beat all individual ML models — here DL architectures are less homogeneous in quality.
3. **Transformer was budget-limited.** max_epochs=15 (CPU/MPS constraint: 14.5s/epoch) left val_rank_IC still improving at epoch 13 for seed 1. More compute budget or a smaller architecture should improve Transformer performance. Best single Transformer seed OOS Sharpe = 2.397, competitive with MLP.
4. **MLP dominates among DL.** Tabular MLP beats both sequence models on MSR wrapping (2.356 vs 2.152 Transformer), suggesting the 63-day lookback sequence adds more noise than signal at this parameter budget. Per-asset processing misses cross-asset structure (Session 3c direction: cross-asset Transformer).

**Session 3c recommendations:**
- Cross-asset Transformer: allow the model to attend across all 29 assets at each time step — the natural DL advantage tree models cannot replicate.
- Increase Transformer compute budget: reduce max_lookback to 63 (vs 512 in pos_embed) to cut parameter count and training cost, or use a smaller d_model=16.
- Consider weighted DL ensemble (upweight MLP) instead of equal-weighted averaging.